# Profiling Agent Runs and Performance Tuning
Companion notebook for the **Profiling Agent Runs and Performance Tuning** section of Chapter 8.

Profile the **full run** -- routing, retrieval, model calls, tool calls, retries, formatting -- not just the final model call. A nested trace with a span per step makes hot spots visible.

A minimal `Span`/`Trace` shape. Only spans that actually have token or retry counts carry those attributes -- `summarize_run` below reads them with `getattr(..., 0)`, exactly so a plain tool span without a `tokens` attribute doesn't blow up:

In [1]:
class Span:
    def __init__(self, name, kind, duration_ms, **extra):
        self.name = name
        self.kind = kind
        self.duration_ms = duration_ms
        for key, value in extra.items():
            setattr(self, key, value)


class Trace:
    def __init__(self, total_ms, spans):
        self.total_ms = total_ms
        self.spans = spans

This is the book's `summarize_run`, unchanged:

In [2]:
def summarize_run(trace):
    return {
        "total_ms": trace.total_ms,
        "slowest_span": max(trace.spans, key=lambda s: s.duration_ms).name,
        "tool_calls": sum(1 for s in trace.spans if s.kind == "tool"),
        "llm_tokens": sum(getattr(s, "tokens", 0) for s in trace.spans),
        "retries": sum(getattr(s, "retries", 0) for s in trace.spans),
    }

A realistic one-request trace: routing (rule-based, cheap), a tool call that had to retry once, an LLM call, and formatting:

In [3]:
trace = Trace(
    total_ms=1840,
    spans=[
        Span("route_request", kind="rule", duration_ms=2),
        Span("policy_lookup", kind="tool", duration_ms=310, retries=1),
        Span("draft_reply", kind="llm", duration_ms=1480, tokens=612),
        Span("format_output", kind="rule", duration_ms=6),
    ],
)

print(summarize_run(trace))

{'total_ms': 1840, 'slowest_span': 'draft_reply', 'tool_calls': 1, 'llm_tokens': 612, 'retries': 1}


`draft_reply` is the slowest span by far (1480ms of 1840ms total) -- that's where tuning effort should go first, not the tool call, even though the tool call is the one that needed a retry. Sort by total time *and* by frequency: a step that's only slightly slow but runs on every request can cost more overall than a rare timeout.

In [4]:
# Sorted view, the way the book suggests reading a trace: total time descending
for span in sorted(trace.spans, key=lambda s: s.duration_ms, reverse=True):
    extra = f" tokens={span.tokens}" if hasattr(span, "tokens") else ""
    extra += f" retries={span.retries}" if hasattr(span, "retries") else ""
    print(f"{span.duration_ms:5d}ms  {span.kind:5s}  {span.name}{extra}")

 1480ms  llm    draft_reply tokens=612
  310ms  tool   policy_lookup retries=1
    6ms  rule   format_output
    2ms  rule   route_request


Then tune one layer at a time -- trim the prompt behind `draft_reply`, cache `policy_lookup` results for stable inputs (notebook 06), or skip the retry entirely if the answer was already in state. Compare **before/after using real sampled traces**, not synthetic examples or a change that just "feels better."